# Task 15: 14-species CV, Final-B and grayscale/16px runs (07-11-2026)

Supervisor requests 6–7 (06-11-2026), on the 14 clean species (17 minus *Callitris columellaris*,
*Cunninghamia lanceolata*, *Fokienia hodginsii*; 4,115 images, 140 origins):

| Request | Runs | Output (`data/experiments/`) |
|---|---:|---|
| 6, CV | 5 models × 5 folds = 25 | `Task-B-clean14-cv/<model>/fold_<k>/` |
| 6, Final-B | 5 models × seeds 42/123/456 = 15 | `Task-B-clean14-final/seed_<s>/<model>/` |
| 7 | ConvNeXt × 5 folds × {grayscale, px16} = 10 | `Task-B-clean14-cv-{grayscale,px16}/` |

Rule for all 50 runs: backbone and head LR 1e-4, 5 epochs linear warmup + cosine, at most 50 epochs,
`best.pth` = lowest val loss from epoch 6 (warmup excluded), early stopping patience 15 on val loss,
held-out fold (CV) or val + test (Final-B) scored once with `best.pth`. CV folds are the CV v2 folds
drawn on 17 species, then the 3 species are dropped (`origin_cv.drop_species`).

This notebook is the record of the run. It starts `run_clean14.sh` (22 jobs, `MAX_PARALLEL` at a time),
prints the launcher events, the latest epoch of every running job, each finished run's result and the
files it wrote, every `POLL_SECONDS`, then builds the tables with `summarize_clean14.py` and copies the
logs into `artifacts/`. Full per-job logs: `data/experiments/clean14-logs/<UTC>/<job>.log`.
Runbook: `docs/timeline/07-11-2026-handoff-vast-clean14.md`.

Run headless (papermill saves the executed copy while a cell runs):

```bash
cd "notebooks/phase 2/supervisor-feedback/task_15_clean14_runs"
papermill clean14_runs.ipynb \
  ../../../../artifacts/phase_2/supervisor-feedback/task_15_clean14_runs/clean14_runs_executed.ipynb \
  -k python3 --cwd .
```

Re-running is safe: finished folds/runs (with `metrics.json`) are kept; unfinished ones restart at epoch 1.

In [1]:
from pathlib import Path
import sys

# SSH server: detect from notebook location (cwd)
ROOT = Path.cwd().parents[3]  # notebooks/phase 2/supervisor-feedback/task_15_clean14_runs/ -> woodbench/
assert (ROOT / "src").is_dir() and (ROOT / "configs").is_dir(), ROOT

sys.path.insert(0, str(ROOT))

import torch
print(f"[ROOT] {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

[ROOT] /root/woodbench
PyTorch: 2.14.1+cu126
CUDA: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 25.3 GB


In [2]:
# Parameters (papermill can override)
MAX_PARALLEL = 5
POLL_SECONDS = 300
PHASES = ""  # "" = cv degraded final; or e.g. "final"

In [3]:
import json
import os
import platform
import re
import shutil
import subprocess
import time
from datetime import datetime, timezone

import pandas as pd
import timm
import torchvision
from IPython.display import Markdown, display

assert torch.cuda.is_available(), "CUDA is required"
TASK_DIR = Path.cwd()
PYTHON = sys.executable
DATA_ROOT = ROOT / "data/task-B-group-split-224"
EXP_ROOT = ROOT / "data/experiments"
ARTIFACTS = ROOT / "artifacts/phase_2/supervisor-feedback/task_15_clean14_runs"
CV_ROOTS = {"rgb": "Task-B-clean14-cv", "grayscale": "Task-B-clean14-cv-grayscale", "px16": "Task-B-clean14-cv-px16"}
FINAL_ROOT = EXP_ROOT / "Task-B-clean14-final"


def sh(*command, cwd=ROOT):
    return subprocess.run(command, cwd=cwd, capture_output=True, text=True).stdout.strip()


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


print("started:", utc_now())
print("commit:", sh("git", "rev-parse", "HEAD"), "| branch:", sh("git", "branch", "--show-current"))
print("git status (tracked):", sh("git", "status", "--short", "--untracked-files=no") or "clean")
print(f"python {platform.python_version()} ({PYTHON})")
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}), torchvision {torchvision.__version__}, timm {timm.__version__}")
print("Note: the handoff names torch 2.15.0+cu132; this host driver supports CUDA 12.7 only, so the")
print("      cu126 build of torch 2.14.1 is used (handoff section 3.2 allows this).")
print(sh("nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"))
print(f"CPU cores: {os.cpu_count()}")
for split in ("train", "val", "test"):
    print(f"{split}: {sum(1 for _ in (DATA_ROOT / split).rglob('*.png'))} images (cache, 17 species)")

started: 2026-11-07 07:06:18 UTC
commit: a496be4cb1d03e56b58b56329df65e29546f15aa | branch: feat/supervisor-2026-11-06
git status (tracked): clean
python 3.12.16 (/venv/wood/bin/python3.12)
torch 2.14.1+cu126 (CUDA 12.6), torchvision 0.29.1+cu126, timm 1.0.30
Note: the handoff names torch 2.15.0+cu132; this host driver supports CUDA 12.7 only, so the
      cu126 build of torch 2.14.1 is used (handoff section 3.2 allows this).
NVIDIA GeForce RTX 4090, 565.57.01, 24564 MiB
CPU cores: 48
train: 3365 images (cache, 17 species)
val: 903 images (cache, 17 species)
test: 532 images (cache, 17 species)


## 1. Start the launcher

`run_clean14.sh` checks the 224 cache against `docs/timeline/01-11-2026-task-B-group-split-224.sha256`,
records commit, `pip freeze` and `nvidia-smi` in its log folder, then queues the jobs. It runs in its own
session, so training continues even if this kernel stops; re-run this notebook to resume monitoring.

In [4]:
class Attached:
    # A launcher started by an earlier run of this notebook (watched by pid).
    def __init__(self, pid):
        self.pid, self.returncode = pid, None

    def poll(self):
        if Path(f"/proc/{self.pid}").exists():
            return None
        status = launch_out.read_text()
        self.returncode = 0 if re.search(r"^All jobs done; 0 failed", status, re.M) else 1
        return self.returncode


running_pids = sh("pgrep", "-f", r"^bash .*/run_clean14\.sh").split()
if running_pids:
    # Never start a second launcher: it would delete runs the first one is training.
    launch_out = sorted((EXP_ROOT / "clean14-logs").glob("launch-*.out"))[-1]
    launcher = Attached(int(running_pids[0]))
    print(f"ATTACHED to running launcher pid {launcher.pid}; stdout -> {launch_out.relative_to(ROOT)}")
else:
    launch_out = EXP_ROOT / "clean14-logs" / f"launch-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.out"
    launch_out.parent.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PYTHON": PYTHON, "MAX_PARALLEL": str(MAX_PARALLEL), "PYTHONUNBUFFERED": "1"}
    launcher = subprocess.Popen(
        ["bash", str(TASK_DIR / "run_clean14.sh"), *PHASES.split()],
        cwd=TASK_DIR, env=env, stdout=open(launch_out, "w"), stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, start_new_session=True,
    )
    print(f"launcher pid {launcher.pid}; stdout -> {launch_out.relative_to(ROOT)}")

LOG_DIR = None
for _ in range(600):
    text = launch_out.read_text()
    match = re.search(r"^Logs: (.+)$", text, re.M)
    if match and "jobs, at most" in text:
        LOG_DIR = Path(match.group(1))
        break
    if launcher.poll() is not None:
        break
    time.sleep(1)
print(launch_out.read_text())
assert LOG_DIR is not None, "launcher stopped before queuing jobs"
JOBS = (LOG_DIR / "jobs.txt").read_text().split()
print(f"log folder: {LOG_DIR.relative_to(ROOT)}")
print((LOG_DIR / "launch.txt").read_text())

launcher pid 6225; stdout -> data/experiments/clean14-logs/launch-20261107T070618Z.out


Data OK: /root/woodbench/data/task-B-group-split-224
Logs: /root/woodbench/data/experiments/clean14-logs/20261107T070619Z
22 jobs, at most 5 at a time
07:06:19 cv_efficientnet_b3 started
07:06:19 cv_resnet50 started
07:06:19 cv_mobilenetv3_large started
07:06:19 cv_swin_tiny started
07:06:19 cv_convnext_tiny started

log folder: data/experiments/clean14-logs/20261107T070619Z
phases: cv degraded final
max_parallel: 5
host: 63da576f4a49
a496be4cb1d03e56b58b56329df65e29546f15aa



## 2. Monitor

Every `POLL_SECONDS`: launcher events, GPU load, latest epoch of each running job (`[best]` = new lowest
val loss, `(warmup)` = epochs 1–5, not eligible), and every run that finished since the last poll with its
result and the files it wrote. Val numbers during training are AMP batch means; the reported ones are in
`metrics.json`.

In [5]:
def run_label(path):
    rel = path.parent.relative_to(EXP_ROOT).parts
    if rel[0] == FINAL_ROOT.name:
        return f"final {rel[1]} {rel[2]}"
    variant = next(k for k, v in CV_ROOTS.items() if v == rel[0])
    return f"cv {variant} {rel[1]} {rel[2]}"


def run_result(path):
    m = json.loads(path.read_text())
    if "test" in m:
        return (f"best epoch {m['best_epoch']}/{m['epochs_trained']}: val acc {m['val']['accuracy'] * 100:.2f}%, "
                f"test acc {m['test']['accuracy'] * 100:.2f}%, test macro-F1 {m['test']['f1_macro']:.4f}")
    return (f"best epoch {m['best_epoch']}/{m['epochs_trained']}: held-out acc {m['accuracy'] * 100:.2f}%, "
            f"macro-F1 {m['f1_macro']:.4f}")


def run_files(folder):
    items = []
    for f in sorted(p for p in folder.iterdir() if p.is_file()):
        size = f.stat().st_size
        items.append(f"{f.name} ({size / 1e6:.0f} MB)" if size > 1e6 else f.name)
    return ", ".join(items)


def finished_runs():
    paths = [p for root in CV_ROOTS.values() for p in (EXP_ROOT / root).glob("*/fold_*/metrics.json")]
    paths += list(FINAL_ROOT.glob("seed_*/*/metrics.json"))
    return set(paths)


def last_epoch_line(job):
    log = LOG_DIR / f"{job}.log"
    if not log.exists():
        return "(no log yet)"
    lines = [l.strip() for l in re.split(r"[\r\n]", log.read_text(errors="replace")) if l.strip()]
    epoch = [l for l in lines if re.search(r"epoch \d+/\d+", l)]
    return epoch[-1] if epoch else (lines[-1][:140] if lines else "(empty log)")


seen_runs = set()
seen_events = 0
start = time.time()
while True:
    done = launcher.poll() is not None
    print(f"===== {utc_now()} | elapsed {(time.time() - start) / 60:.0f} min =====")
    events = launch_out.read_text().splitlines()
    events = [e for e in events if re.match(r"\d\d:\d\d:\d\d ", e) or e.startswith("All jobs")]
    for event in events[seen_events:]:
        print("  launcher:", event)
    seen_events = len(events)
    exits = {p.stem: p.read_text().strip() for p in LOG_DIR.glob("*.exit")}
    started = {e.split()[1] for e in events if e.endswith(" started")}
    running = [j for j in JOBS if j in started and j not in exits]
    failed = sorted(j for j, s in exits.items() if s != "0")
    gpu = sh("nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader")
    print(f"  jobs: {len(exits)}/{len(JOBS)} done, {len(running)} running, {len(failed)} failed {failed or ''}"
          f" | runs with metrics.json: {len(finished_runs())}/50 | GPU {gpu}")
    for job in running:
        print(f"  {job:>28}: {last_epoch_line(job)}")
    for path in sorted(finished_runs() - seen_runs):
        print(f"  NEW {run_label(path)}: {run_result(path)}")
        print(f"      {path.parent.relative_to(ROOT)}/: {run_files(path.parent)}")
    seen_runs = finished_runs()
    if done:
        break
    time.sleep(POLL_SECONDS)
print(f"launcher exit code {launcher.returncode}")

===== 2026-11-07 07:06:19 UTC | elapsed 0 min =====
  launcher: 07:06:19 cv_efficientnet_b3 started
  launcher: 07:06:19 cv_resnet50 started
  launcher: 07:06:19 cv_mobilenetv3_large started
  launcher: 07:06:19 cv_swin_tiny started
  launcher: 07:06:19 cv_convnext_tiny started
  jobs: 0/22 done, 5 running, 0 failed  | runs with metrics.json: 0/50 | GPU 0 %, 4 MiB, 24564 MiB
            cv_efficientnet_b3: (empty log)
                   cv_resnet50: (empty log)
          cv_mobilenetv3_large: (empty log)
                  cv_swin_tiny: (empty log)
              cv_convnext_tiny: (empty log)


===== 2026-11-07 07:11:19 UTC | elapsed 5 min =====
  jobs: 0/22 done, 5 running, 0 failed  | runs with metrics.json: 0/50 | GPU 100 %, 16391 MiB, 24564 MiB
            cv_efficientnet_b3: Fold 1, epoch 19/50: train=0.9939 val=0.9832 loss=0.6692 [best]
                   cv_resnet50: Fold 1, epoch 30/50: train=0.9899 val=0.9760 loss=0.6084
          cv_mobilenetv3_large: Fold 1, epoch 50/50: train=0.9992 val=0.9772 loss=0.6922 [best]
                  cv_swin_tiny: Fold 1, epoch 21/50: train=0.9882 val=0.9388 loss=0.7124
              cv_convnext_tiny: Fold 1, epoch 22/50: train=1.0000 val=0.9940 loss=0.5617


===== 2026-11-07 07:16:19 UTC | elapsed 10 min =====
  jobs: 0/22 done, 5 running, 0 failed  | runs with metrics.json: 4/50 | GPU 100 %, 16327 MiB, 24564 MiB
            cv_efficientnet_b3: Fold 1, epoch 41/50: train=0.9972 val=0.9832 loss=0.6334
                   cv_resnet50: Fold 2, epoch 10/50: train=0.9420 val=0.9134 loss=0.7951
          cv_mobilenetv3_large: Fold 2, epoch 50/50: train=0.9959 val=0.9514 loss=0.7475
                  cv_swin_tiny: Fold 1, epoch 42/50: train=1.0000 val=0.9964 loss=0.5558
              cv_convnext_tiny: Fold 2, epoch 5/50: train=0.9812 val=0.9632 loss=0.6442
  NEW cv rgb convnext_tiny fold_1: best epoch 25/40: held-out acc 99.26%, macro-F1 0.9930
      data/experiments/Task-B-clean14-cv/convnext_tiny/fold_1/: best.pth (111 MB), history.json, metrics.json, predictions.csv, run.json
  NEW cv rgb mobilenetv3_large fold_1: best epoch 50/50: held-out acc 93.87%, macro-F1 0.9365
      data/experiments/Task-B-clean14-cv/mobilenetv3_large/fold_1/: best.pth 

===== 2026-11-07 07:21:19 UTC | elapsed 15 min =====
  jobs: 0/22 done, 5 running, 0 failed  | runs with metrics.json: 8/50 | GPU 100 %, 16335 MiB, 24564 MiB
            cv_efficientnet_b3: Fold 2, epoch 12/50: train=0.9828 val=0.9395 loss=0.7699 [best]
                   cv_resnet50: Fold 2, epoch 41/50: train=0.9947 val=0.9561 loss=0.6610
          cv_mobilenetv3_large: Fold 4, epoch 2/50: train=0.1859 val=0.2225 loss=3.0079
                  cv_swin_tiny: Fold 2, epoch 14/50: train=0.9984 val=0.9893 loss=0.5812 [best]
              cv_convnext_tiny: Fold 3, epoch 4/50: train=0.9898 val=0.8785 loss=0.8473
  NEW cv rgb convnext_tiny fold_2: best epoch 10/25: held-out acc 99.15%, macro-F1 0.9921
      data/experiments/Task-B-clean14-cv/convnext_tiny/fold_2/: best.pth (111 MB), history.json, metrics.json, predictions.csv, run.json
  NEW cv rgb efficientnet_b3 fold_1: best epoch 50/50: held-out acc 95.96%, macro-F1 0.9590
      data/experiments/Task-B-clean14-cv/efficientnet_b3/fold_1/: 

===== 2026-11-07 07:26:20 UTC | elapsed 20 min =====
  jobs: 0/22 done, 5 running, 0 failed  | runs with metrics.json: 11/50 | GPU 100 %, 14363 MiB, 24564 MiB
            cv_efficientnet_b3: Fold 2, epoch 34/50: train=0.9988 val=0.9703 loss=0.6642
                   cv_resnet50: Fold 3, epoch 22/50: train=0.9829 val=0.9607 loss=0.6576 [best]
          cv_mobilenetv3_large: Fold 5, epoch 3/50: train=0.4912 val=0.5400 loss=1.8759
                  cv_swin_tiny: Fold 2, epoch 35/50: train=1.0000 val=0.9917 loss=0.5741
              cv_convnext_tiny: Fold 3, epoch 29/50: train=0.9976 val=0.9902 loss=0.5747
  NEW cv rgb convnext_tiny fold_3: best epoch 14/29: held-out acc 99.65%, macro-F1 0.9962
      data/experiments/Task-B-clean14-cv/convnext_tiny/fold_3/: best.pth (111 MB), history.json, metrics.json, predictions.csv, run.json
  NEW cv rgb mobilenetv3_large fold_4: best epoch 50/50: held-out acc 97.42%, macro-F1 0.9725
      data/experiments/Task-B-clean14-cv/mobilenetv3_large/fold_4/: b

===== 2026-11-07 07:31:20 UTC | elapsed 25 min =====
  launcher: 07:30:41 cv_mobilenetv3_large finished (exit 0)
  launcher: 07:30:41 cv_grayscale_convnext_tiny started
  jobs: 1/22 done, 5 running, 0 failed  | runs with metrics.json: 15/50 | GPU 100 %, 17099 MiB, 24564 MiB
            cv_efficientnet_b3: Fold 3, epoch 5/50: train=0.8924 val=0.8380 loss=1.0407
                   cv_resnet50: Fold 4, epoch 2/50: train=0.0897 val=0.1064 loss=2.6220
                  cv_swin_tiny: Fold 3, epoch 8/50: train=0.9882 val=0.9521 loss=0.6911
              cv_convnext_tiny: Fold 4, epoch 24/50: train=1.0000 val=0.9988 loss=0.5503
    cv_grayscale_convnext_tiny: Fold 1, epoch 1/50: train=0.1042 val=0.0792 loss=2.8702
  NEW cv rgb efficientnet_b3 fold_2: best epoch 48/50: held-out acc 97.69%, macro-F1 0.9768
      data/experiments/Task-B-clean14-cv/efficientnet_b3/fold_2/: best.pth (43 MB), history.json, metrics.json, predictions.csv, run.json
  NEW cv rgb mobilenetv3_large fold_5: best epoch 38/5

... (10 outputs left out)

===== 2026-11-07 08:26:21 UTC | elapsed 80 min =====
  launcher: 08:23:31 final_s123_mobilenetv3_large finished (exit 0)
  launcher: 08:23:31 final_s123_convnext_tiny started
  launcher: 08:24:14 final_s123_efficientnet_b3 finished (exit 0)
  launcher: 08:24:14 final_s456_efficientnet_b3 started
  jobs: 13/22 done, 5 running, 0 failed  | runs with metrics.json: 41/50 | GPU 100 %, 17031 MiB, 24564 MiB
         cv_px16_convnext_tiny: Fold 5, epoch 14/50: train=0.9668 val=0.8900 loss=0.8345 [best]
           final_s123_resnet50: [resnet50 s123] epoch 46/50: train=0.9941 val=0.9726 loss=0.6149
          final_s123_swin_tiny: [swin_tiny s123] epoch 20/50: train=0.9955 val=0.9888 loss=0.5814
      final_s123_convnext_tiny: [convnext_tiny s123] epoch 10/50: train=0.9868 val=0.9564 loss=0.6891
    final_s456_efficientnet_b3: [efficientnet_b3 s456] epoch 6/50: train=0.9594 val=0.9427 loss=0.7038 [best]
  NEW cv px16 convnext_tiny fold_4: best epoch 32/47: held-out acc 91.39%, macro-F1 0.9079
  

===== 2026-11-07 08:31:21 UTC | elapsed 85 min =====
  launcher: 08:27:12 final_s123_resnet50 finished (exit 0)
  launcher: 08:27:12 final_s456_resnet50 started
  jobs: 14/22 done, 5 running, 0 failed  | runs with metrics.json: 42/50 | GPU 100 %, 17031 MiB, 24564 MiB
         cv_px16_convnext_tiny: Fold 5, epoch 36/50: train=1.0000 val=0.9163 loss=0.7690
          final_s123_swin_tiny: [swin_tiny s123] epoch 38/50: train=1.0000 val=0.9975 loss=0.5540
      final_s123_convnext_tiny: [convnext_tiny s123] epoch 31/50: train=1.0000 val=0.9913 loss=0.5675
    final_s456_efficientnet_b3: [efficientnet_b3 s456] epoch 24/50: train=0.9979 val=0.9763 loss=0.6092
           final_s456_resnet50: [resnet50 s456] epoch 19/50: train=0.9844 val=0.9577 loss=0.6500 [best]
  NEW final seed_123 resnet50: best epoch 49/50: val acc 98.13%, test acc 97.45%, test macro-F1 0.9759
      data/experiments/Task-B-clean14-final/seed_123/resnet50/: best.pth (94 MB), history.json, metrics.json, predictions_test.csv, 

===== 2026-11-07 08:36:22 UTC | elapsed 90 min =====
  launcher: 08:32:30 final_s123_convnext_tiny finished (exit 0)
  launcher: 08:32:30 final_s456_mobilenetv3_large started
  launcher: 08:33:38 cv_px16_convnext_tiny finished (exit 0)
  launcher: 08:33:38 final_s456_swin_tiny started
  launcher: 08:34:46 final_s123_swin_tiny finished (exit 0)
  launcher: 08:34:46 final_s456_convnext_tiny started
  jobs: 17/22 done, 5 running, 0 failed  | runs with metrics.json: 45/50 | GPU 100 %, 16285 MiB, 24564 MiB
    final_s456_efficientnet_b3: [efficientnet_b3 s456] epoch 44/50: train=0.9990 val=0.9801 loss=0.5931
           final_s456_resnet50: [resnet50 s456] epoch 46/50: train=0.9965 val=0.9851 loss=0.6072
  final_s456_mobilenetv3_large: [mobilenetv3_large s456] epoch 36/50: train=0.9983 val=0.9738 loss=0.6116
          final_s456_swin_tiny: [swin_tiny s456] epoch 9/50: train=0.9948 val=0.9813 loss=0.5926
      final_s456_convnext_tiny: [convnext_tiny s456] epoch 5/50: train=0.9771 val=0.9228 

===== 2026-11-07 08:41:22 UTC | elapsed 95 min =====
  launcher: 08:36:41 final_s456_efficientnet_b3 finished (exit 0)
  launcher: 08:37:01 final_s456_resnet50 finished (exit 0)
  launcher: 08:37:04 final_s456_mobilenetv3_large finished (exit 0)
  launcher: 08:38:46 final_s456_convnext_tiny finished (exit 0)
  launcher: 08:40:16 final_s456_swin_tiny finished (exit 0)
  launcher: All jobs done; 0 failed (see *.exit and logs in /root/woodbench/data/experiments/clean14-logs/20261107T070619Z)
  jobs: 22/22 done, 0 running, 0 failed  | runs with metrics.json: 50/50 | GPU 0 %, 4 MiB, 24564 MiB
  NEW final seed_456 convnext_tiny: best epoch 10/25: val acc 99.88%, test acc 99.07%, test macro-F1 0.9917
      data/experiments/Task-B-clean14-final/seed_456/convnext_tiny/: best.pth (111 MB), history.json, metrics.json, predictions_test.csv, predictions_val.csv, run_config.json
  NEW final seed_456 efficientnet_b3: best epoch 30/45: val acc 98.51%, test acc 97.45%, test macro-F1 0.9767
      data/e

In [6]:
exits = {p.stem: p.read_text().strip() for p in LOG_DIR.glob("*.exit")}
status = pd.DataFrame({"job": JOBS, "exit": [exits.get(j, "missing") for j in JOBS]})
print(status.to_string(index=False))
bad = status[status.exit != "0"]
for job in bad.job:
    print(f"\n--- tail of {job}.log ---")
    print("\n".join((LOG_DIR / f"{job}.log").read_text(errors="replace").splitlines()[-30:]))
assert bad.empty, f"{len(bad)} job(s) failed; fix and re-run this notebook (finished runs are kept)"
assert len(finished_runs()) == 50, len(finished_runs())
print("All 22 jobs exited 0; 50 runs have metrics.json")

                         job exit
          cv_efficientnet_b3    0
                 cv_resnet50    0
        cv_mobilenetv3_large    0
                cv_swin_tiny    0
            cv_convnext_tiny    0
  cv_grayscale_convnext_tiny    0
       cv_px16_convnext_tiny    0
   final_s42_efficientnet_b3    0
          final_s42_resnet50    0
 final_s42_mobilenetv3_large    0
         final_s42_swin_tiny    0
     final_s42_convnext_tiny    0
  final_s123_efficientnet_b3    0
         final_s123_resnet50    0
final_s123_mobilenetv3_large    0
        final_s123_swin_tiny    0
    final_s123_convnext_tiny    0
  final_s456_efficientnet_b3    0
         final_s456_resnet50    0
final_s456_mobilenetv3_large    0
        final_s456_swin_tiny    0
    final_s456_convnext_tiny    0
All 22 jobs exited 0; 50 runs have metrics.json


## 3. Tables

`summarize_clean14.py` exports the CV reports (registry, OOF, per-species, as for CV v2) and writes
`artifacts/phase_2/supervisor-feedback/task_15_clean14_runs/results/`. SD is the sample SD over folds
or seeds; it describes variation, not a confidence interval. One Final-B test image = 0.23 points (432
images); differences of 2–3 images are noise.

In [7]:
result = subprocess.run([PYTHON, "summarize_clean14.py"], cwd=TASK_DIR, capture_output=True, text=True)
print(result.stdout[-6000:])
print(result.stderr[-3000:])
assert result.returncode == 0, "summarize_clean14.py failed"
RESULTS = ARTIFACTS / "results"
display(Markdown((RESULTS / "summary.md").read_text()))

# 14-species runs (supervisor requests 6-7, 06-11-2026)

CV report `Task-B-clean14-cv`: `cv-20261107T084125Z-5a586a35`

CV report `Task-B-clean14-cv-grayscale`: `cv-20261107T084125Z-0fa52558`

CV report `Task-B-clean14-cv-px16`: `cv-20261107T084126Z-9f17c7a8`

## CV, 5 models × 5 folds (held-out fold, best.pth)

| Model | Acc (%) | Macro-F1 (%) | OOF acc (%) | Best epoch | Epochs |
|---|---:|---:|---:|---:|---:|
| convnext_tiny | 99.44 ± 0.31 | 99.46 ± 0.28 | 99.44 | 17.4 | 32.4 |
| swin_tiny | 99.34 ± 0.26 | 99.33 ± 0.24 | 99.34 | 33.4 | 47.2 |
| efficientnet_b3 | 97.57 ± 0.93 | 97.51 ± 0.92 | 97.57 | 48.0 | 50.0 |
| resnet50 | 97.12 ± 1.35 | 97.02 ± 1.33 | 97.13 | 39.2 | 49.6 |
| mobilenetv3_large | 95.29 ± 1.34 | 94.91 ± 1.45 | 95.29 | 46.4 | 50.0 |

## Same folds, same 14 species: 17-species model (CV v2, 01-11 rule) vs 14-species model

CV v2 accuracy counts a prediction of one of the 3 removed species as wrong.

| Model | CV v2 on 14 species (%) | 14-species CV (%) | Mean differe

# 14-species runs (supervisor requests 6-7, 06-11-2026)

CV report `Task-B-clean14-cv`: `cv-20261107T084125Z-5a586a35`

CV report `Task-B-clean14-cv-grayscale`: `cv-20261107T084125Z-0fa52558`

CV report `Task-B-clean14-cv-px16`: `cv-20261107T084126Z-9f17c7a8`

## CV, 5 models × 5 folds (held-out fold, best.pth)

| Model | Acc (%) | Macro-F1 (%) | OOF acc (%) | Best epoch | Epochs |
|---|---:|---:|---:|---:|---:|
| convnext_tiny | 99.44 ± 0.31 | 99.46 ± 0.28 | 99.44 | 17.4 | 32.4 |
| swin_tiny | 99.34 ± 0.26 | 99.33 ± 0.24 | 99.34 | 33.4 | 47.2 |
| efficientnet_b3 | 97.57 ± 0.93 | 97.51 ± 0.92 | 97.57 | 48.0 | 50.0 |
| resnet50 | 97.12 ± 1.35 | 97.02 ± 1.33 | 97.13 | 39.2 | 49.6 |
| mobilenetv3_large | 95.29 ± 1.34 | 94.91 ± 1.45 | 95.29 | 46.4 | 50.0 |

## Same folds, same 14 species: 17-species model (CV v2, 01-11 rule) vs 14-species model

CV v2 accuracy counts a prediction of one of the 3 removed species as wrong.

| Model | CV v2 on 14 species (%) | 14-species CV (%) | Mean difference (pts) |
|---|---:|---:|---:|
| convnext_tiny | 99.03 ± 0.41 | 99.44 ± 0.31 | +0.41 |
| swin_tiny | 98.88 ± 1.21 | 99.34 ± 0.26 | +0.47 |
| efficientnet_b3 | 97.47 ± 1.14 | 97.57 ± 0.93 | +0.09 |
| resnet50 | 96.89 ± 1.27 | 97.12 ± 1.35 | +0.23 |
| mobilenetv3_large | 96.24 ± 1.17 | 95.29 ± 1.34 | -0.95 |

## ConvNeXt-Tiny, input degradation (same 5 folds)

| Input | Acc per fold (%) | Acc (%) | Macro-F1 (%) | Best epoch |
|---|---|---:|---:|---:|
| rgb | 99.26, 99.15, 99.65, 99.88, 99.26 | 99.44 ± 0.31 | 99.46 ± 0.28 | 17.4 |
| grayscale | 91.79, 93.69, 95.16, 96.31, 97.18 | 94.83 ± 2.14 | 94.50 ± 2.23 | 27.8 |
| px16 | 90.32, 91.38, 89.61, 91.39, 91.78 | 90.90 ± 0.90 | 90.13 ± 1.24 | 34.6 |

## Final-B, 5 models × 3 seeds (test 432 images, best.pth)

| Model | Seeds | Test acc (%) | Test macro-F1 (%) | Val acc (%) | Best epoch per seed |
|---|---:|---:|---:|---:|---|
| convnext_tiny | 3 | 98.92 ± 0.27 | 98.99 ± 0.29 | 99.63 ± 0.43 | 18, 20, 10 |
| swin_tiny | 3 | 99.69 ± 0.13 | 99.72 ± 0.12 | 99.67 ± 0.26 | 44, 40, 22 |
| efficientnet_b3 | 3 | 97.92 ± 0.46 | 98.07 ± 0.41 | 98.30 ± 0.59 | 31, 43, 30 |
| resnet50 | 3 | 97.53 ± 0.81 | 97.67 ± 0.82 | 98.30 ± 0.19 | 39, 49, 49 |
| mobilenetv3_large | 3 | 98.53 ± 0.27 | 98.64 ± 0.27 | 98.51 ± 0.33 | 46, 46, 29 |

50 checkpoints listed in `checkpoints.sha256`.


In [8]:
pd.set_option("display.width", 200, "display.max_columns", 30)
for name in ("cv_summary", "cv_vs_17species", "degraded_summary", "final_b_summary", "final_b_runs"):
    print(f"\n### {name}.csv")
    display(pd.read_csv(RESULTS / f"{name}.csv"))


### cv_summary.csv


,backbone,n_folds,accuracy_mean,accuracy_sd,f1_macro_mean,f1_macro_sd,oof_accuracy,best_epoch_mean,epochs_trained_mean
0,convnext_tiny,5,0.994404,0.003078,0.994636,0.002778,0.994411,17.4,32.4
1,swin_tiny,5,0.993432,0.002562,0.993285,0.002429,0.993439,33.4,47.2
2,efficientnet_b3,5,0.975661,0.009305,0.975056,0.009177,0.975699,48.0,50.0
3,resnet50,5,0.971233,0.013529,0.970226,0.013267,0.971324,39.2,49.6
4,mobilenetv3_large,5,0.952912,0.013370,0.949090,0.014469,0.952855,46.4,50.0



### cv_vs_17species.csv


,backbone,fold,cv_v2_17species_on_14,n_images,clean14,n_clean14,difference
0,convnext_tiny,1,0.990196,816,0.992647,816,0.002451
1,convnext_tiny,2,0.992718,824,0.991505,824,-0.001214
2,convnext_tiny,3,0.983471,847,0.996458,847,0.012987
3,convnext_tiny,4,0.993850,813,0.998770,813,0.004920
4,convnext_tiny,5,0.991411,815,0.992638,815,0.001227
5,swin_tiny,1,0.985294,816,0.991422,816,0.006127
6,swin_tiny,2,0.995146,824,0.995146,824,0.000000
7,swin_tiny,3,0.994097,847,0.994097,847,0.000000
8,swin_tiny,4,1.000000,813,0.996310,813,-0.003690
9,swin_tiny,5,0.969325,815,0.990184,815,0.020859



### degraded_summary.csv


,backbone,input_variant,fold,best_epoch,epochs_trained,best_val_loss,best_val_accuracy,n_test,accuracy,f1_macro,run_id
0,convnext_tiny,rgb,1,25,40,0.560108,0.993998,816,0.992647,0.992962,convnext_tiny-s42-f1-8ee690df2aaf43bc924ecc663...
1,convnext_tiny,rgb,2,10,25,0.589106,0.985765,824,0.991505,0.992110,convnext_tiny-s42-f2-b88cf4b2e29c46bea669490d9...
2,convnext_tiny,rgb,3,14,29,0.568138,0.992638,847,0.996458,0.996227,convnext_tiny-s42-f3-bb766e53df6b4cc4abb8b29ea...
3,convnext_tiny,rgb,4,25,40,0.549843,0.998791,813,0.998770,0.998748,convnext_tiny-s42-f4-e6b905978fa94c4f9020252aa...
4,convnext_tiny,rgb,5,13,28,0.561923,0.996250,815,0.992638,0.993135,convnext_tiny-s42-f5-2e92304e6bfb4ee2b99b872b4...
5,convnext_tiny,grayscale,1,28,43,0.602831,0.969988,816,0.917892,0.913898,convnext_tiny-s42-f1-9c232bd9fda1458eb28204aca...
6,convnext_tiny,grayscale,2,27,42,0.691830,0.947805,824,0.936893,0.934646,convnext_tiny-s42-f2-2d3e18778cc3434a8dd2845a9...
7,convnext_tiny,grayscale,3,28,43,0.611761,0.975460,847,0.951594,0.945652,convnext_tiny-s42-f3-82a069613ef94662931fc7650...
8,convnext_tiny,grayscale,4,21,36,0.610075,0.970979,813,0.963100,0.959151,convnext_tiny-s42-f4-b5be5587085d434e81e2be6eb...
9,convnext_tiny,grayscale,5,35,50,0.619127,0.972500,815,0.971779,0.971804,convnext_tiny-s42-f5-d52e49149dd24df495c04eae0...



### final_b_summary.csv


,backbone,n_seeds,test_accuracy_mean,test_accuracy_sd,test_f1_macro_mean,test_f1_macro_sd,val_accuracy_mean,val_accuracy_sd
0,convnext_tiny,3,0.989198,0.002673,0.989949,0.002929,0.996264,0.004314
1,swin_tiny,3,0.996914,0.001336,0.997195,0.001213,0.996679,0.002592
2,efficientnet_b3,3,0.979167,0.004630,0.980702,0.004142,0.982980,0.005885
3,resnet50,3,0.975309,0.008129,0.976714,0.008177,0.982980,0.001902
4,mobilenetv3_large,3,0.985340,0.002673,0.986366,0.002723,0.985056,0.003295



### final_b_runs.csv


,backbone,seed,best_epoch,epochs_trained,val_loss,val_accuracy,val_f1_macro,test_accuracy,test_f1_macro,test_errors,n_test,run_id
0,convnext_tiny,42,18,33,0.552757,0.998755,0.998847,0.986111,0.986568,6,432,convnext_tiny-s42-clean14-636e4ffd343443c986b3...
1,convnext_tiny,123,20,35,0.564840,0.991283,0.990909,0.990741,0.991597,4,432,convnext_tiny-s123-clean14-4ff6edf182f54c5787a...
2,convnext_tiny,456,10,25,0.557417,0.998755,0.998809,0.990741,0.991684,4,432,convnext_tiny-s456-clean14-82b64e6a57324f9a937...
3,swin_tiny,42,44,50,0.550378,0.998755,0.998703,0.997685,0.997895,1,432,swin_tiny-s42-clean14-f62b3136df504e3980b17616...
4,swin_tiny,123,40,50,0.551169,0.997509,0.997414,0.997685,0.997895,1,432,swin_tiny-s123-clean14-8bfeb81c0aef4ca0b1d4bf9...
5,swin_tiny,456,22,37,0.562826,0.993773,0.993366,0.995370,0.995795,2,432,swin_tiny-s456-clean14-cbb087243533421d8a06954...
6,efficientnet_b3,42,31,46,0.603235,0.976339,0.974212,0.983796,0.985005,7,432,efficientnet_b3-s42-clean14-48ac453548f44eab82...
7,efficientnet_b3,123,43,50,0.577540,0.987547,0.987104,0.979167,0.980359,9,432,efficientnet_b3-s123-clean14-0d0aab6252064295b...
8,efficientnet_b3,456,30,45,0.587998,0.985056,0.984654,0.974537,0.976742,11,432,efficientnet_b3-s456-clean14-a53f57239f9246ebb...
9,resnet50,42,39,50,0.598857,0.982565,0.981609,0.983796,0.985265,7,432,resnet50-s42-clean14-c993ea4c43624dab9eacfbf4c...


## 4. Keep the logs with the artifacts

Copies `data/experiments/clean14-logs/` (launcher output, per-job logs and exit codes, `pip_freeze.txt`,
`nvidia-smi.txt`) to `artifacts/.../task_15_clean14_runs/logs/` and lists what goes into git.
Checkpoints (`best.pth`) stay out of git; `results/checkpoints.sha256` lists them and they are backed up
to Drive (`drive:wood/2026-11-07-clean14-runs/`, handoff section 6.3).

In [9]:
shutil.copytree(EXP_ROOT / "clean14-logs", ARTIFACTS / "logs", dirs_exist_ok=True)
rows = []
for f in sorted(p for p in ARTIFACTS.rglob("*") if p.is_file()):
    rows.append({"file": str(f.relative_to(ARTIFACTS)), "kB": round(f.stat().st_size / 1e3, 1)})
listing = pd.DataFrame(rows)
print(f"{len(listing)} files, {listing.kB.sum() / 1e3:.1f} MB under {ARTIFACTS.relative_to(ROOT)}")
top = listing.assign(group=listing.file.str.split("/").str[:2].str.join("/")).groupby("group").agg(files=("file", "size"), kB=("kB", "sum"))
display(top)
n_ckpt = len((RESULTS / "checkpoints.sha256").read_text().splitlines())
ckpt_gb = sum(p.stat().st_size for p in EXP_ROOT.glob("Task-B-clean14-*/**/best.pth")) / 1e9
print(f"checkpoints.sha256: {n_ckpt} entries; best.pth on disk: {ckpt_gb:.1f} GB")
print("finished:", utc_now())

307 files, 33.3 MB under artifacts/phase_2/supervisor-feedback/task_15_clean14_runs


,files,kB
group,,
clean14_runs_executed.ipynb,1,108.0
logs/20261107T070619Z,48,11404.1
logs/launch-20261107T070618Z.out,1,2.2
logs/papermill.log,1,43.5
results/checkpoints.sha256,1,6.7
results/cv_folds.csv,1,4.0
results/cv_summary.csv,1,0.7
results/cv_vs_17species.csv,1,2.1
results/degraded_summary.csv,1,2.5


checkpoints.sha256: 50 entries; best.pth on disk: 4.1 GB
finished: 2026-11-07 08:41:29 UTC
